In [57]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Check headers

In [74]:
# Read lines 8 and 9 to get both potential header rows
header_rows = pd.read_excel('../../Data prep/Bestand/02 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Herstellern und Handelsnamen (FZ 2)/fz2_2024.xlsx', 
                           sheet_name='FZ 2.2', 
                           skiprows=6, 
                           nrows=2)

# Get the headers from both rows
header_row1 = header_rows.iloc[0]
header_row2 = header_rows.iloc[1]

# Combine headers where line 9 has values
combined_headers = []
for col1, col2 in zip(header_row1, header_row2):
    if pd.notna(col2):  # If there's a value in line 9
        combined_headers.append(f"{col1}: {col2}")
    else:
        combined_headers.append(col1)

# Now read the full file with the combined headers
df = pd.read_excel('../../Data prep/Bestand/02 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Herstellern und Handelsnamen (FZ 2)/fz2_2024.xlsx', 
                   sheet_name='FZ 2.2', 
                   header=7)  # Start from line 8

# Apply the combined headers
df.columns = combined_headers

# Drop the first row since we've already used it for headers
df = df[1:].reset_index(drop=True)

In [75]:
df.head()

,NaN,Hersteller\n,Handelsname,Typ-Schl.-Nr.,kW,Kraftstoffart,Allrad,Aufbauart,Insgesamt,Und zwar: Wohnmobile,nan: private Halterinnen\nund Halter,nan: Halterinnen \nund Halter\nbis 29 Jahre,nan: Halterinnen\nund Halter\nab 60 Jahre,nan: Halterinnen
0,NaN,ALPINA,BMW ALPINA B3 BITURBO,ABK,301.0,B,A,NaN,139.0,-,126,4,49,17
1,NaN,NaN,BMW ALPINA B3 Touring,ACV,340.0,B,A,NaN,238.0,-,120,1,61,11
2,NaN,NaN,NaN,ADS,364.0,B,A,NaN,104.0,-,44,-,23,5
3,NaN,NaN,BMW ALPINA B5,ACX,457.0,B,A,NaN,303.0,-,115,2,53,15
4,NaN,NaN,BMW ALPINA B5 BITURBO,ACJ,447.0,B,A,NaN,125.0,-,75,2,46,7


## Old code: Use line 8 and if empty than line 9
The logic is actually not what we need

In [ ]:
# First read lines 8 and 9 to check which one contains the header
header_check = pd.read_excel('../../Data prep/Bestand/02 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Herstellern und Handelsnamen (FZ 2)/fz2_2024.xlsx', 
                            sheet_name='FZ 2.2', 
                            skiprows=7, 
                            nrows=2)

In [ ]:
# Check if line 8 (first row) is empty
if header_check.iloc[1].isna().all():
    header_row = 8  # Use line 9 as header
else:
    header_row = 7  # Use line 8 as header

# print(f"Using row {header_row + 1} as header")

In [ ]:
# Now read the full file with the correct header row
df = pd.read_excel('../../Data prep/Bestand/02 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Herstellern und Handelsnamen (FZ 2)/fz2_2024.xlsx', 
                   sheet_name='FZ 2.2', 
                   header=header_row)

In [ ]:
# Check if there are any unnamed columns
if df.columns.str.contains('^Unnamed').any():
    first_row = df.iloc[0]

    # Replace only the unnamed headers
    new_columns = [
        first_row[i] if col.startswith('Unnamed') else col
        for i, col in enumerate(df.columns)
    ]

    df.columns = new_columns
    df = df[1:].reset_index(drop=True)  # Drop the first row (used as partial header)

## Now clean headers

In [76]:
# Adjust header format
df.columns = df.columns.str.replace('\n', ' ', regex=False)
df.columns = df.columns.str.replace('\t', ' ', regex=False)
df.columns = df.columns.str.strip()
# Check the columns
# print(df.columns.tolist())

In [78]:
# Replace "nan:" -> with "Und zwar:""
df.columns = df.columns.str.replace('nan:', 'Und zwar:')

In [79]:
df.head()

,NaN,Hersteller,Handelsname,Typ-Schl.-Nr.,kW,Kraftstoffart,Allrad,Aufbauart,Insgesamt,Und zwar: Wohnmobile,Und zwar: private Halterinnen und Halter,Und zwar: Halterinnen und Halter bis 29 Jahre,Und zwar: Halterinnen und Halter ab 60 Jahre,Und zwar: Halterinnen
0,NaN,ALPINA,BMW ALPINA B3 BITURBO,ABK,301.0,B,A,NaN,139.0,-,126,4,49,17
1,NaN,NaN,BMW ALPINA B3 Touring,ACV,340.0,B,A,NaN,238.0,-,120,1,61,11
2,NaN,NaN,NaN,ADS,364.0,B,A,NaN,104.0,-,44,-,23,5
3,NaN,NaN,BMW ALPINA B5,ACX,457.0,B,A,NaN,303.0,-,115,2,53,15
4,NaN,NaN,BMW ALPINA B5 BITURBO,ACJ,447.0,B,A,NaN,125.0,-,75,2,46,7


## Clean Sum lines

In [ ]:
# Drop empty columns add the beginning of the file
df = df.iloc[:, 1:]

In [ ]:
# Delete rows that only contain sums
# Define words to filter out
filter_words = 'zusammen|insgesamt'

# Filter all columns in the dataframe
for column in df.columns:
    df = df[~df[column].astype(str).str.lower().str.contains(filter_words.lower(), case=False, na=False)].reset_index(drop=True) 

## Deal with merged Columns in values
When several lines are in one Bundesland, but Excel only has 1 value that is shown in a merged block

In [ ]:
# Add all columns by name that should be checked / have merged cells -> Change the name of the data file -> should we define and automate that at the beginning?
columns_to_fill = ['Hersteller', 'Handelsname']  
for col in columns_to_fill:
    df[col] = df[col].fillna(method='ffill')

# Display the results
# print("First 10 rows of the DataFrame:")
# print(df[columns_to_fill + ['Unnamed: 3']].head(10))

In [ ]:
df.iloc[13500:13550,]

## Seperate meged values in columns

In [ ]:
def extract_columns(df, source_column, target_columns, first_part_length, is_numeric=True):
    """
    Extract and split data from a source column into target columns.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to modify
    source_column : str
        The source column name containing the data to split
    target_columns : str or list
        The target column name(s) for the extracted data
    first_part_length : int
        Number of characters for the first part
    is_numeric : bool
        If True, first part is numeric (\\d), if False, first part is letters ([A-ZÄÖÜa-zäöü])
    
    Returns:
    --------
    pandas.DataFrame
        The modified dataframe
    """
    # Convert single target column to list for consistent handling
    if isinstance(target_columns, str):
        target_columns = [target_columns]
    
    # Create pattern based on whether we're extracting one or two columns
    if len(target_columns) == 1:
        # For single column extraction, remove the first part
        pattern_type = '\\d' if is_numeric else '[A-ZÄÖÜa-zäöü]'
        pattern = f'^{pattern_type}{{{first_part_length}}}\\s+'
        df[target_columns[0]] = df[source_column].str.replace(pattern, '', regex=True)
    else:
        # For two column extraction, capture both parts
        pattern_type = '\\d' if is_numeric else '[A-ZÄÖÜa-zäöü]'
        pattern = f'({pattern_type}{{{first_part_length}}})\\s+(.+)'
        df[target_columns] = df[source_column].str.extract(pattern)
    
    return df

# Example usage:
# For numeric first part (5 digits) and two columns:
# df_2 = extract_columns(df_2, 'Statistische Kennziffer und Zulassungsbezirk', 
#                       ['Statistische Kennziffer', 'Zulassungsbezirk'], 5, is_numeric=True)

# For letter first part (2 letters) and single column:
# df_2 = extract_columns(df_2, 'Regierungsbezirk', 'Regierungs_Bezirk', 2, is_numeric=False) 